In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os, sys


def get_dir_n_levels_up(path, n):
    # Go up n levels from the given path
    for _ in range(n):
        path = os.path.dirname(path)
    return path

proj_root = get_dir_n_levels_up(os.path.abspath("__file__"), 3)
sys.path.append(proj_root)

In [5]:
import time
from opinion_dynamics.experiments.online import run_multi_seed_experiment_dynamics

def timed_one(seed, dynamics):
    t0 = time.time()
    df = run_multi_seed_experiment_dynamics(
        seeds=[seed],
        dynamics_model=dynamics,
        B_campaign=1.0,
        num_campaigns_total=5,
        suppress_fit_logs=True,
    )
    dt = time.time() - t0
    print(f"[{dynamics}] seed={seed} total_time={dt:.2f}s")
    return df

timed_one(0, "laplacian")
timed_one(0, "coca")

[laplacian] seed=0 total_time=10.69s
[coca] seed=0 total_time=194.02s


,seed,dynamics,N,v_L1_final,A_Fro_final,A_MAE_final,mean_gap_to_oracle_end,mean_err_avg,mean_err_max,vx_gap_to_oracle_end,vx_err_avg,vx_err_max,mean_oracle_end,mean_learn_end,mean_noc_end
0,0,coca,15,0.671183,3.40532,0.108428,0.011837,0.005584,0.011837,0.01971,0.00905,0.01971,0.605295,0.593458,0.289706


In [3]:
# =========================================================
# Multi-seed experiment (10 seeds) — collect metrics only
# =========================================================

import matplotlib.pyplot as plt
import pandas as pd

from opinion_dynamics.experiments.online import run_multi_seed_experiment_dynamics

seeds = range(10)
B_campaign = 1.0
num_campaigns_total = 5

df_lap = run_multi_seed_experiment_dynamics(
    seeds=seeds,
    dynamics_model="laplacian",
    B_campaign=B_campaign,
    num_campaigns_total=num_campaigns_total,
    suppress_fit_logs=True,
)

df_coca = run_multi_seed_experiment_dynamics(
    seeds=seeds,
    dynamics_model="coca",
    B_campaign=B_campaign,
    num_campaigns_total=num_campaigns_total,
    suppress_fit_logs=True,
)

df = pd.concat([df_lap, df_coca], axis=0).reset_index(drop=True)
display(df)

print("\n=== Aggregate by dynamics ===")
cols = ["v_L1_final","A_MAE_final","mean_gap_to_oracle_end","vx_gap_to_oracle_end"]
display(df.groupby("dynamics")[cols].describe().T)

# Simple comparison plot
plt.figure(figsize=(9,3))
for dyn in ["laplacian", "coca"]:
    sub = df[df["dynamics"] == dyn].sort_values("seed")
    plt.plot(sub["seed"], sub["A_MAE_final"], marker="o", label=f"{dyn}: A_MAE_final")
plt.xlabel("seed"); plt.ylabel("MAE(A_hat, A_true)")
plt.title("Adjacency MAE vs seed (laplacian vs coca)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(9,3))
for dyn in ["laplacian", "coca"]:
    sub = df[df["dynamics"] == dyn].sort_values("seed")
    plt.plot(sub["seed"], sub["v_L1_final"], marker="o", label=f"{dyn}: v_L1_final")
plt.xlabel("seed"); plt.ylabel("||v_hat - v_true||_1")
plt.title("Centrality error vs seed (laplacian vs coca)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

C:\Users\Chainsword\AppData\Local\Temp\ipykernel_36324\161128283.py:6: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx

from rl_envs_forge.envs.network_graph.graph_utils import compute_laplacian, compute_eigenvector_centrality

from opinion_dynamics.experiments.plots import (
    plot_impulse_node_trajectories,
    show_matrix_with_cell_grid,
    concat_intermediate,
)
from opinion_dynamics.experiments.metrics import graph_sanity
from opinion_dynamics.experiments.rollouts import rollout_with_v_intermediate
from opinion_dynamics.experiments.online import run_single_seed_experiment


# =========================================================
# Settings
# =========================================================
SEED_TO_PLOT = 6
B_campaign = 1.0
num_campaigns_total = 5


# =========================================================
# 1) Run ONE experiment (online learner)
# =========================================================
metrics, art = run_single_seed_experiment(
    seed=SEED_TO_PLOT,
    B_campaign=B_campaign,
    num_campaigns_total=num_campaigns_total,
    lr=1e-3,
    l2_lambda=0.0,
    device="cpu",
    update_A_each_campaign=True,
    suppress_fit_logs=True,
    return_artifacts=True,
)

print("=== METRICS ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

env_learn = art["env"]  # NOTE: already stepped inside the experiment


# =========================================================
# 2) Pull ground-truth + learned-final
# =========================================================
A_true = np.asarray(art["A_true"], dtype=float)
v_true = np.asarray(art["v_true"], dtype=float)

A_hat_final = np.asarray(art["A_hat_final"], dtype=float)
v_hat_final = np.asarray(art["v_hat_final"], dtype=float)

states_learn = np.asarray(art["states_learn"], dtype=float)  # (K+1, N)
x0 = np.asarray(art["x0"], dtype=float)
N = states_learn.shape[1]
K_total = states_learn.shape[0] - 1  # campaigns actually run


# =========================================================
# 3) Reconstruct a FRESH env for rollouts (same graph/params)
#    This avoids “campaign 0 drift” from reusing a stepped env.
# =========================================================
EnvCls = env_learn.__class__

# Best: if your experiment already returns kwargs used to create env, use them
if "env_template_kwargs" in art:
    kwargs = dict(art["env_template_kwargs"])
else:
    # Fallback: reconstruct from the learned env's attributes
    # (works if NetworkGraph accepts these kwargs)
    kwargs = dict(
        connectivity_matrix=np.array(env_learn.connectivity_matrix, copy=True),
        num_agents=int(env_learn.num_agents),
        max_u=np.array(env_learn.max_u, copy=True),
        desired_opinion=float(env_learn.desired_opinion),
        t_campaign=float(env_learn.t_campaign),
        t_s=float(env_learn.t_s),
        dynamics_model=str(env_learn.dynamics_model),
        control_resistance=np.array(env_learn.control_resistance, copy=True),
        max_steps=int(getattr(env_learn, "max_steps", 10_000)),
        opinion_end_tolerance=float(getattr(env_learn, "opinion_end_tolerance", 0.01)),
        control_beta=float(getattr(env_learn, "control_beta", 0.4)),
        normalize_reward=bool(getattr(env_learn, "normalize_reward", False)),
        terminal_reward=float(getattr(env_learn, "terminal_reward", 0.0)),
        terminate_when_converged=bool(getattr(env_learn, "terminate_when_converged", True)),
        seed=int(getattr(env_learn, "seed", SEED_TO_PLOT)) if getattr(env_learn, "seed", None) is not None else None,
    )

env_base = EnvCls(**kwargs)


# =========================================================
# 4) GRAPH SANITY (true vs learned-final)
# =========================================================
print("\n=== GRAPH SANITY (TRUE) ===")
for k, v in graph_sanity(A_true).items():
    print(f"{k}: {v}")

print("\n=== GRAPH SANITY (LEARNED FINAL) ===")
for k, v in graph_sanity(A_hat_final).items():
    print(f"{k}: {v}")


# =========================================================
# 5) Heatmaps (true vs learned-final) — keep your nicer grid version
# =========================================================
show_matrix_with_cell_grid(A_true,      title=f"True A (seed={SEED_TO_PLOT})", grid_alpha=0.25, grid_lw=0.6)
show_matrix_with_cell_grid(A_hat_final, title=f"Learned A_hat final (seed={SEED_TO_PLOT})", grid_alpha=0.25, grid_lw=0.6)


# =========================================================
# 6) Fine-grained trajectories for LEARNED run (actual)
# =========================================================
inter_list = art["intermediate_states_list"]   # list length K, each (T_k, N)
time_list  = art["intermediate_times_list"]    # list length K, each (T_k,)

valid_pairs = [(xs, ts) for xs, ts in zip(inter_list, time_list) if xs is not None and ts is not None]
if not valid_pairs:
    raise RuntimeError("No intermediate trajectories found. env.step must return info['intermediate_states'].")

dt = float(getattr(env_learn, "t_s", 1.0))

X_le, T_le = concat_intermediate(
    inter_list,
    time_list,
    dt=dt,
)


# =========================================================
# 7) Oracle + no-control rollouts WITH intermediate (fresh env, same x0)
#    IMPORTANT: zero_first_campaign=True ensures campaign 0 matches the paper free-fall.
# =========================================================
or_out = rollout_with_v_intermediate(
    env_base, x0, K_total, B_campaign, v_true, zero_first_campaign=True
)
nc_out = rollout_with_v_intermediate(
    env_base, x0, K_total, B_campaign, None,   zero_first_campaign=True
)

X_or, T_or = concat_intermediate(
    or_out["intermediate_states_list"],
    or_out["intermediate_times_list"],
    dt=dt,
)

# Optional: no-control impulse plot if you want it too
# X_nc, T_nc = concat_intermediate(
#     nc_out["intermediate_states_list"],
#     nc_out["intermediate_times_list"],
#     dt=dt,
# )


# =========================================================
# 8) Impulse-style plots (what actually happens inside campaigns)
# =========================================================
plot_impulse_node_trajectories(X_or, T_or, title=f"Node trajectories (fine-grained) — ORACLE true v (seed={SEED_TO_PLOT})")
plot_impulse_node_trajectories(X_le, T_le, title=f"Node trajectories (fine-grained) — LEARNED (seed={SEED_TO_PLOT})")


# =========================================================
# 9) Centrality comparison (NO sorting; x-axis = node index)
# =========================================================
print("\n=== CENTRALITY COMPARISON (by node index) ===")
print("L1(v_hat_final - v_true):", float(np.sum(np.abs(v_hat_final - v_true))))

idx = np.arange(N)
diff = v_hat_final - v_true

plt.figure(figsize=(10, 4))
plt.plot(idx, v_true, marker="o", label="v_true")
plt.plot(idx, v_hat_final, marker="o", label="v_hat_final")
plt.xlabel("node index i")
plt.ylabel("centrality value")
plt.title("Centrality comparison (true vs learned) — by node index")
plt.xticks(idx)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

plt.figure(figsize=(10, 3))
plt.axhline(0.0, linewidth=1)
plt.plot(idx, diff, marker="o")
plt.xlabel("node index i")
plt.ylabel("v_hat - v_true")
plt.title("Centrality error per node — by node index")
plt.xticks(idx)
plt.grid(True, alpha=0.3)
plt.show()


# =========================================================
# 10) Mean opinion plot (campaign boundaries)
# =========================================================
states_oracle = np.asarray(or_out["states"], dtype=float)
states_noc    = np.asarray(nc_out["states"], dtype=float)

mean_learn = states_learn.mean(axis=1)
mean_or    = states_oracle.mean(axis=1)
mean_nc    = states_noc.mean(axis=1)

plt.figure(figsize=(8, 3))
plt.plot(mean_learn, marker="o", label="learned online control")
plt.plot(mean_or,    marker="o", label="oracle (true v)")
plt.plot(mean_nc,    marker="o", label="no control")
plt.xlabel("Campaign boundary k")
plt.ylabel("mean opinion")
plt.title("Trajectory: mean opinion (campaign boundaries)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.ylim(0, 1)
plt.show()


# =========================================================
# 11) Quick consistency check: campaign 0 should overlap (oracle vs learned vs nocontrol)
# =========================================================
print("\n=== CAMPAIGN-0 OVERLAP CHECK ===")
print("learned states[1] vs oracle states[1] L_inf:", float(np.max(np.abs(states_learn[1] - states_oracle[1]))))
print("learned states[1] vs nocontrol states[1] L_inf:", float(np.max(np.abs(states_learn[1] - states_noc[1]))))